In [1]:
import numpy as np
import pandas as pd
import torch
import os
import time
import openai
import pickle

path ='/n/data1/hsph/biostat/celehs/lab/hat127/GAME_0527//'+'/'
os.chdir(path+'/src')

from config import config
from utils import id_map, ret_type, write_file, all_fea_select, feature_selection_every_epoch
from sklearn.metrics import roc_auc_score
from evaluate import test
CHECK_ALL=0

def match(a, b, rm_None=True):
    # Create an array to store the indices
    indices = np.array([np.where(b == x)[0][0] if x in b else np.nan for x in a])
    if rm_None:
        # Filter out None values, which represent elements not found in b
        indices = indices[~np.isnan(indices)]
    return indices

def match_table(RELA_VA, RELA):
    matched_indices = match(RELA[0][0].index, RELA_VA[0][0].index, rm_None=False)
    if np.isnan(matched_indices).any():
        # If there are any NaN values in the indices, replace them with a new row filled with NaNs
        return pd.concat([RELA_VA[0][0].iloc[matched_indices].fillna(np.nan), pd.DataFrame(np.nan, index=np.where(np.isnan(matched_indices)), columns=RELA_VA[0][0].columns)], ignore_index=True)
    else:
        # If all indices are integers, no change
        return RELA_VA[0][0].iloc[matched_indices]

def compute_weighted_ave(df):
    df = df[0][0]
    df = df.dropna(subset=['auc', 'num'])
    weighted_average_auc = np.average(df['auc'], weights=df['num'])
    return np.round(weighted_average_auc,3)

def tidy_test(x0, name_all, config, sampled_REL, sampled_SIM, AUC_type=False, PRE=True):
    if PRE is True:
        PRE, SIMI, RELA = test(x0, name_all, config, sampled_REL, sampled_SIM, AUC_type=AUC_type, PRE=PRE)
    else:
        SIMI, RELA = test(x0, name_all, config, sampled_REL, sampled_SIM, AUC_type=AUC_type, PRE=PRE) 
    SIMI = compute_weighted_ave(SIMI)
    RELA = compute_weighted_ave(RELA)
    print(f'similarity AUC = {SIMI}')
    print(f'relatedness AUC = {RELA}')
    return PRE, SIMI, RELA

In [2]:
path_dir = '/n/data1/hsph/biostat/celehs/lab/hat127/GAME_ablation_all_2//'

x1, x2, x3, x4, x5, x6, x7 = torch.load(f'{config["input_dir"]}/emb/inst_emb.pth')

x8 = torch.load(f'{config["input_dir"]}/emb/sap_emb.pth')
x9 = torch.load(f'{config["input_dir"]}/emb/coder_emb.pth')
x10 = torch.load(f'{config["input_dir"]}/emb/bge_emb.pth')
x11 = torch.load(f'{config["input_dir"]}/emb/openai_emb.pth')

xa = torch.load(path_dir+'/GAME_hie/output/2025-06-06 00:12:22/baseline_emb_81.pth')  # a
xb = torch.load(path_dir+'/GAME_hie_umls/output/2025-06-07 18:32:01/baseline_emb_101.pth')  #  b
xd = torch.load(path_dir+'/GAME_GATS_wloss/output/2025-06-06 09:25:05/baseline_emb_51.pth')  #  b
xe = torch.load(path_dir+'/GAME_better_feature_selection/output/2025-06-07 17:43:58/baseline_emb_31.pth')  # e
xf = torch.load(path_dir+'/GAME_better_all//output/2025-06-08 18:09:12/baseline_emb_51.pth')  # f
xg = torch.load(path_dir+'/GAME_only_GPT/output/2025-06-07 18:37:04/baseline_emb_51.pth')  # g
xh = torch.load('/n/data1/hsph/biostat/celehs/lab/hat127/GAME_ablation_all/GAME_no_contrastive2/output/2025-06-06 21:01:50/rel_emb.pth')
GAME = torch.load('/n/data1/hsph/biostat/celehs/lab/hat127/GAME_0527/output/2025-06-02 13:50:53/rel_emb_40.pth')

BIOBERT = pd.read_csv(f'{config["input_dir"]}/emb/biobert_emb.csv')
BIOBERT = BIOBERT.values.astype('float32')
BIOBERT = torch.tensor(BIOBERT)

PUBMED = pd.read_csv(f'{config["input_dir"]}/emb/pubmedbert_emb.csv')
PUBMED = PUBMED.values.astype('float32')
PUBMED = torch.tensor(PUBMED)

unique_name = pd.read_csv(f'{config["input_dir"]}/name_desc/unique_name_desc.csv')
name_all = unique_name.iloc[:,0].values

data = np.load(f'{config["input_dir"]}/name_desc/inst_row.npz')
keys = data.files  # This will give you a list of all keys in the .npz file
config['inst_row'] = [data[key] for key in keys]

In [3]:
# config['inst_row']
# MGB, VA, UPMC, BCH, Duke, MIMIC, BDX

[array([   0,    1,    2, ..., 6966, 6967, 6968]),
 array([    0,     1,     2, ..., 10319, 10320, 10321]),
 array([    0,     1,     2, ..., 22169, 22170, 22171]),
 array([    0,     1,     2, ..., 23565, 23566, 23567]),
 array([  227,   228,   246, ..., 24488, 24489, 24490]),
 array([    0,     1,     2, ..., 28059, 28060, 28061]),
 array([  228,   234,   247, ..., 50735, 50736, 50737])]

In [ ]:
x_all = [x4, x7, x5, x1, x6, x3, x2, BIOBERT, PUBMED, x8, x9, x10, x11, GAME  ]
name_list = ['BCH', 'BDX','Duke','MGB','MIMIC','UPMC','VA', 'BBERT', 'PBERT', 'SBERT', 'CODER', 'BGE', 'OpenAI', 'GAME' ]

config['path'] = '/n/data1/hsph/biostat/celehs/lab/hat127/GAME_0527/'

code_list = ["PheCode:428.1", "PheCode:296.2", "PheCode:714.1", "PheCode:290.11", "PheCode:250.1", "PheCode:250.2", "PheCode:555.1", "PheCode:555.2"]
loc = 'RESULT0610'
epoch = 0
api_key = 'XXXX' # your OPENAI API KEYS 

In [ ]:
corr_list, result_list = feature_selection_every_epoch(x_all, loc=loc, epoch=epoch, name_list=name_list, code_list = code_list, api_key=api_key, RECORD=-1, config=config)